# Create HEC Pakistan NRPU Awards

Creates OpenAlex award rows from the Higher Education Commission, Pakistan official NRPU project outcomes portal.

- Funder: Higher Education Commission, Pakistan (`F4320322799`; ROR `https://ror.org/038y3sz68`; DOI `10.13039/501100004681`)
- Source authority: HEC Research Grant Statistics page links to the NRPU outcomes portal; the portal exposes a same-origin DataTables endpoint.
- Source method: official source ladder 3/5 hybrid, public DataTables endpoint behind a static official page.
- Local validation on 2026-05-27: 1,892 rows, 1,892 distinct project numbers, 100% title/project/university/status coverage, 99.9% PI name coverage.
- Amount/currency/year: NULL by source limitation; the endpoint does not publish per-project money or award dates.
- Provenance: `hec_pakistan_nrpu`; priority: `140`.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hec_pakistan_nrpu_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hec_pakistan/hec_pakistan_nrpu.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 1,892 rows.
SELECT COUNT(*) AS total_rows
FROM openalex.awards.hec_pakistan_nrpu_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.hec_pakistan_nrpu_raw;


In [ ]:
%sql
-- Sample raw HEC Pakistan NRPU data.
SELECT
    funder_award_id,
    project_number,
    display_name,
    pi_name,
    university,
    status,
    major_field,
    outcomes_pdf_url
FROM openalex.awards.hec_pakistan_nrpu_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys


In [ ]:
%sql
-- Money/date-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'hec_pakistan_nrpu_raw'
  AND LOWER(column_name) RLIKE 'amount|amt|currency|year|date|total|fund|grant|award'
ORDER BY column_name;


In [ ]:
%sql
-- Coverage profile from the official DataTables endpoint.
SELECT
    COUNT(*) AS total_rows,
    COUNT(project_number) AS project_number_rows,
    ROUND(try_divide(COUNT(project_number), COUNT(*)) * 100.0, 1) AS pct_project_number,
    COUNT(display_name) AS title_rows,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    COUNT(pi_name) AS pi_name_rows,
    ROUND(try_divide(COUNT(pi_name), COUNT(*)) * 100.0, 1) AS pct_pi_name,
    COUNT(pi_email) AS pi_email_rows,
    ROUND(try_divide(COUNT(pi_email), COUNT(*)) * 100.0, 1) AS pct_pi_email,
    COUNT(university) AS university_rows,
    ROUND(try_divide(COUNT(university), COUNT(*)) * 100.0, 1) AS pct_university,
    COUNT(status) AS status_rows,
    ROUND(try_divide(COUNT(status), COUNT(*)) * 100.0, 1) AS pct_status,
    COUNT(outcomes_pdf_url) AS outcomes_url_rows,
    ROUND(try_divide(COUNT(outcomes_pdf_url), COUNT(*)) * 100.0, 1) AS pct_outcomes_url,
    COUNT(major_field) AS major_field_rows,
    ROUND(try_divide(COUNT(major_field), COUNT(*)) * 100.0, 1) AS pct_major_field,
    COUNT(amount) AS amount_rows,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(source_year) AS source_year_rows,
    ROUND(try_divide(COUNT(source_year), COUNT(*)) * 100.0, 1) AS pct_source_year
FROM openalex.awards.hec_pakistan_nrpu_raw;


In [ ]:
%sql
-- Native-key inspection. Project number is unique and is used in funder_award_id.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT project_number) AS distinct_project_numbers,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids
FROM openalex.awards.hec_pakistan_nrpu_raw;


In [ ]:
%sql
-- Must return zero rows.
SELECT funder_award_id, COUNT(*) AS rows
FROM openalex.awards.hec_pakistan_nrpu_raw
GROUP BY funder_award_id
HAVING COUNT(*) > 1
ORDER BY rows DESC, funder_award_id;


In [ ]:
%sql
-- Status and field distribution.
SELECT
    status,
    major_field,
    COUNT(*) AS rows
FROM openalex.awards.hec_pakistan_nrpu_raw
GROUP BY status, major_field
ORDER BY rows DESC, status, major_field
LIMIT 50;


In [ ]:
%sql
-- Amount/currency waiver diagnostic: HEC's public endpoint does not publish per-project money.
SELECT
    funder_scheme,
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(currency) AS rows_with_currency,
    ROUND(try_divide(COUNT(currency), COUNT(*)) * 100.0, 1) AS pct_currency
FROM openalex.awards.hec_pakistan_nrpu_raw
GROUP BY funder_scheme
ORDER BY rows DESC;


## Step 1.6: Funder Existence Fail-Fast


In [ ]:
%sql
-- Must return one OK row. If this fails, STOP: the canonical HEC Pakistan funder is missing.
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for HEC Pakistan F4320322799'
) AS funder_exists
FROM openalex.common.funder
WHERE funder_id = 4320322799;


In [ ]:
%sql
-- Diagnostic funder row. OpenAlex also has an older misspelled duplicate; this ingest uses the canonical ROR-backed row.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320322799;


## Step 2: Transform to OpenAlex Award Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hec_pakistan_awards
USING delta
AS
WITH
hec_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320322799
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        NULLIF(TRIM(currency), '') AS parsed_currency,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_year
    FROM openalex.awards.hec_pakistan_nrpu_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS id,

        TRIM(g.display_name) AS display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END AS description,

        f.funder_id,
        g.native_award_id AS funder_award_id,

        g.parsed_amount AS amount,
        g.parsed_currency AS currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,

        'grant' AS funding_type,

        COALESCE(NULLIF(TRIM(g.funder_scheme), ''), 'National Research Program for Universities (NRPU)') AS funder_scheme,

        'hec_pakistan_nrpu' AS provenance,

        g.parsed_start_date AS start_date,
        g.parsed_end_date AS end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_year) AS start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_year) AS end_year,

        CASE
            WHEN g.pi_name IS NULL OR TRIM(g.pi_name) = '' THEN CAST(NULL AS STRUCT<
                given_name:STRING,
                family_name:STRING,
                orcid:STRING,
                role_start:DATE,
                affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
            >)
            ELSE struct(
                NULLIF(TRIM(g.pi_given_name), '') AS given_name,
                NULLIF(TRIM(g.pi_family_name), '') AS family_name,
                CAST(NULL AS STRING) AS orcid,
                g.parsed_start_date AS role_start,
                struct(
                    NULLIF(TRIM(g.university), '') AS name,
                    'PK' AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        END AS lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,

        NULLIF(TRIM(g.landing_page_url), '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) AS works_api_url,

        current_timestamp() AS created_date,
        current_timestamp() AS updated_date

    FROM raw_prepared g
    CROSS JOIN hec_funder f
)

SELECT * FROM awards_transformed;


In [ ]:
%sql
-- Check transformed row count and ID uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_native_ids
FROM openalex.awards.hec_pakistan_awards;


In [ ]:
%sql
SELECT
    id,
    display_name,
    funder_award_id,
    funder.display_name AS funder_name,
    funding_type,
    funder_scheme,
    lead_investigator.given_name AS pi_given_name,
    lead_investigator.family_name AS pi_family_name,
    lead_investigator.affiliation.name AS pi_affiliation,
    amount,
    currency,
    start_year,
    landing_page_url
FROM openalex.awards.hec_pakistan_awards
LIMIT 20;


## Step 3: Delete Existing Rows and Insert Fresh Priority 140 Rows


In [ ]:
%sql
-- Remove previous HEC Pakistan NRPU data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hec_pakistan_nrpu' AND priority = 140;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    140 AS priority
FROM openalex.awards.hec_pakistan_awards;


In [ ]:
%sql
-- Confirm inserted row count.
SELECT
    provenance,
    priority,
    COUNT(*) AS rows,
    COUNT(DISTINCT funder_award_id) AS distinct_native_ids,
    COUNT(DISTINCT id) AS distinct_openalex_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hec_pakistan_nrpu'
GROUP BY provenance, priority;


## Step 6: Verification Queries


In [ ]:
%sql
-- Duplicate OpenAlex IDs within this ingest: must return zero rows.
SELECT id, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hec_pakistan_nrpu' AND priority = 140
GROUP BY id
HAVING COUNT(*) > 1
ORDER BY rows DESC, id;


In [ ]:
%sql
-- Field coverage in the inserted award rows.
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS title_rows,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    COUNT(funder_award_id) AS native_id_rows,
    ROUND(try_divide(COUNT(funder_award_id), COUNT(*)) * 100.0, 1) AS pct_native_id,
    COUNT(lead_investigator.family_name) AS pi_family_rows,
    ROUND(try_divide(COUNT(lead_investigator.family_name), COUNT(*)) * 100.0, 1) AS pct_pi_family,
    COUNT(lead_investigator.affiliation.name) AS pi_affiliation_rows,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) AS pct_pi_affiliation,
    COUNT(landing_page_url) AS landing_url_rows,
    ROUND(try_divide(COUNT(landing_page_url), COUNT(*)) * 100.0, 1) AS pct_landing_url,
    COUNT(amount) AS amount_rows,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(start_year) AS start_year_rows,
    ROUND(try_divide(COUNT(start_year), COUNT(*)) * 100.0, 1) AS pct_start_year
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hec_pakistan_nrpu' AND priority = 140;


In [ ]:
%sql
-- Amount/currency waiver verification by scheme.
SELECT
    funder_scheme,
    COUNT(*) AS rows,
    COUNT(amount) AS amount_rows,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(currency) AS currency_rows,
    ROUND(try_divide(COUNT(currency), COUNT(*)) * 100.0, 1) AS pct_currency
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hec_pakistan_nrpu' AND priority = 140
GROUP BY funder_scheme
ORDER BY rows DESC;


In [ ]:
%sql
-- Funder and provenance sanity check.
SELECT
    funder_id,
    funder.display_name AS funder_name,
    provenance,
    priority,
    funding_type,
    COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hec_pakistan_nrpu' AND priority = 140
GROUP BY funder_id, funder.display_name, provenance, priority, funding_type;


In [ ]:
%sql
-- Sample records for human QA.
SELECT
    id,
    display_name,
    description,
    funder_award_id,
    lead_investigator,
    amount,
    currency,
    start_year,
    landing_page_url,
    works_api_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hec_pakistan_nrpu' AND priority = 140
ORDER BY funder_award_id
LIMIT 25;


## Handoff / Admin Notes

Contractor-complete handoff only. The script and notebook are ready, but the contractor has no S3 or Databricks access.

Admin must:

- Run `scripts/local/hec_pakistan_to_s3.py` without `--skip-upload` to upload `hec_pakistan_nrpu.parquet`.
- Run this Databricks notebook.
- Inspect the Step 6 verification cells, especially the source-limited amount/year NULLs.
- Run `CreateAwards.ipynb` after QA.
- Flip the tracker row from Step 4 to Complete only after production verification.
